# AI enabled summarization

## Prerequisites

We assume you have access to both Azure OpenAI and have already deployed an LLM.

## Get started

In this tutorial we will be using Azure OpenAI which (if you havent already deployed) you can learn how to deploy [here](https://github.com/STRIDES/NIHCloudLabAzure/blob/main/notebooks/GenAI/azure_infra_setup/README.md). This tutorial utilizes the model gpt-5.4-nano and the embeddings model text-embedding-3-small.

## Setting up Python and Azure OpenAI environment

In [ ]:
# azure cloud login
!az login --identity

In [ ]:
# set your subscription
!az account set --subscription "NIH.CIT.CS.CloudLab.Azure_XXX" # your subscription name

In [ ]:
# confirm subscription
!az account show

In [ ]:
%pip install --quiet openai

### Get Azure OpenAI environment from Azure
In search bar window, type **Foundry** and select **Microsoft Foundry**

In microsoft foundry window, click on **resource** where you deployed gpt and embedding models

For Azure OpenAI endpoint, click rectangle under **Azure OpenAI endpoint**. It would copy OpenAI endpoint, replace it in below cell with **your_openai_endpoint**

For Azure OpenAI Key, click rectangle under **API key**. It would copy OpenAI key, replace it in below cell with **yourOpenAIKey**

In [ ]:
# configure environment
import os, time, json, numpy as np, pandas as pd
from typing import List, Dict
from xml.etree import ElementTree as ET
from openai import OpenAI
import requests

AZURE_OPENAI_ENDPOINT="your_openai_endpoint"
AZURE_OPENAI_KEY = "your_openAI_key"
AZURE_OPENAI_DEPLOYMENT_EMBED =  "text-embedding-3-small"
AZURE_OPENAI_DEPLOYMENT_CHAT =  "gpt-5.4-nano"

client = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_KEY)

## Utility Functions
Basic cleaning, chunking, and cosine similarity for RAG.

In [ ]:

import re
import math
import numpy as np
from typing import List

def clean_text(s: str) -> str:
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

def chunk_text(text: str, size: int = 1500, overlap: int = 200) -> List:
    text = clean_text(text)
    chunks = []
    i = 0
    while i < len(text):
        end = i + size
        chunks.append(text[i:end])
        i = end - overlap
        if i < 0: i = 0
    return chunks

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    return float(np.dot(a, b) / denom) if denom else 0.0


## Translation (Chat Completions)
A minimal translation example using your chat deployment.

In [ ]:

prompt = "Translate this sentence from English to French: Why all the hype about Generative AI?"
resp = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT_CHAT,
    messages=[{"role": "user", "content": prompt}]
)

print(resp.choices[0].message.content)


## Load PubMed Article (Requests + BeautifulSoup)
If the abstract isn't available, the loader will fall back to the entire page text.

In [ ]:

import requests
from bs4 import BeautifulSoup

PUBMED_URL = "https://pubmed.ncbi.nlm.nih.gov/37883540/"
html = requests.get(PUBMED_URL, timeout=30).text
soup = BeautifulSoup(html, "html.parser")

# Try to extract abstract; fall back to the whole page if not found
abstract_div = soup.find("div", class_="abstract-content")
text = clean_text(abstract_div.get_text(" ")) if abstract_div else clean_text(soup.get_text(" "))

print("Preview:", text[:600], "...")


## Map–Reduce Summarization (Chat only)
Summarize each chunk (map), then combine partial summaries (reduce).

In [ ]:

# Split text into chunks
chunks = chunk_text(text, size=1500, overlap=200)
print(f"Number of chunks: {len(chunks)}")

# ---- Map step: summarize each chunk
partials = []
for i, chunk in enumerate(chunks, 1):
    r = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_CHAT,
        messages=[
            {"role": "system", "content": "You summarize scientific text clearly and factually."},
            {"role": "user", "content": f"Summarize this scientific text chunk:\n\n{chunk}"}
        ]
    )
   
    partials.append(r.choices[0].message.content)
    if i % 3 == 0:
        print(f"Summarized chunk {i}/{len(chunks)}")

# ---- Reduce step: combine partial summaries
final = client.chat.completions.create(
    model=AZURE_OPENAI_DEPLOYMENT_CHAT,
    messages=[
        {
            "role": "system",
            "content": "You are an expert scientific writer. Combine partial summaries into a single, structured abstract. Include: Background, Methods, Results, Conclusions."
        },
        {
            "role": "user",
            "content": "Combine the following partial summaries into one high‑quality scientific summary:\n\n" + "\n\n".join(partials)
        }
    ]
)
summary = final.choices[0].message.content
print("\n=== Final Summary ===\n", summary)


## Embeddings + Simple RAG (NumPy cosine similarity)
Compute embeddings for each chunk, rank by cosine similarity, and answer with top‑k context.

In [ ]:

# Create embeddings for each chunk
chunk_vecs = []
for i, chunk in enumerate(chunks, 1):
    e = client.embeddings.create(
        model=AZURE_OPENAI_DEPLOYMENT_EMBED,
        input=chunk
    )
    vec = np.array(e.data[0].embedding, dtype=np.float32)
    chunk_vecs.append(vec)
    if i % 5 == 0:
        print(f"Embedded chunk {i}/{len(chunks)}")

# Function: answer a question using top-k chunks as context
def answer_question(question: str, k: int = 5) -> str:
    # Embed the question
    q = client.embeddings.create(
        model=AZURE_OPENAI_DEPLOYMENT_EMBED,
        input=question
    )
    qvec = np.array(q.data[0].embedding, dtype=np.float32)

    # Rank chunks by cosine similarity
    sims = [cosine_sim(qvec, v) for v in chunk_vecs]
    top_idx = np.argsort(sims)[-k:][::-1]  # top-k indices

    context = "\n\n".join(chunks[i] for i in top_idx)
    # Ask the chat model using retrieved context
    resp = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_CHAT,
        messages=[
            {"role": "system", "content": "Answer using only the provided context. If uncertain, say 'insufficient context'."},
            {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"}
        ]
    )
    return resp.choices[0].message.content

# Example query
print("\n=== RAG Answer ===\n", answer_question("What is the main finding of the study?"))
